In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [2]:
from nltk.corpus import stopwords
nltk.download('stopwords')
import string
from nltk.stem.snowball import SnowballStemmer
sb = SnowballStemmer('english')
import re

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
heading = ['id', 'entity' , 'sentiment' , 'tweet content']

In [5]:
df = pd.read_csv('/content/drive/MyDrive/sahil nawaz alam/twitter_training.csv',header=None,names=heading)

In [6]:
df.head()

,id,entity,sentiment,tweet content
0,2401,Borderlands,Positive,im getting on borderlands and i will murder yo...
1,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
2,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
3,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
4,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...


In [7]:
df = df.drop('id', axis=1)

In [8]:
df.head()

,entity,sentiment,tweet content
0,Borderlands,Positive,im getting on borderlands and i will murder yo...
1,Borderlands,Positive,I am coming to the borders and I will kill you...
2,Borderlands,Positive,im getting on borderlands and i will kill you ...
3,Borderlands,Positive,im coming on borderlands and i will murder you...
4,Borderlands,Positive,im getting on borderlands 2 and i will murder ...


In [9]:
df.entity.value_counts()

,count
entity,
Microsoft,2400
MaddenNFL,2400
TomClancysRainbowSix,2400
LeagueOfLegends,2394
CallOfDuty,2394
Verizon,2382
CallOfDutyBlackopsColdWar,2376
ApexLegends,2376
Facebook,2370


In [10]:
df.sentiment.value_counts()

,count
sentiment,
Negative,22542
Positive,20832
Neutral,18318
Irrelevant,12990


In [11]:
df['sentiment'].replace({'Negative': 0,'Positive': 1 , "Neutral":2 , "Irrelevant":3},inplace=True)

/tmp/ipykernel_4985/3748719121.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['sentiment'].replace({'Negative': 0,'Positive': 1 , "Neutral":2 , "Irrelevant":3},inplace=True)
/tmp/ipykernel_4985/3748719121.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['sentiment'].replace({'Negative'

In [12]:
df.sentiment.value_counts()

,count
sentiment,
0,22542
1,20832
2,18318
3,12990


In [13]:
G = df[df["sentiment"]==3]

In [ ]:
G

,entity,sentiment,tweet content
102,Borderlands,3,Appreciate the (sonic) concepts / praxis Valen...
103,Borderlands,3,Appreciate the (sound) concepts / practices th...
104,Borderlands,3,Evaluate the (sound) concepts / concepts of Va...
105,Borderlands,3,Appreciate the (sonic) concepts / praxis Valen...
106,Borderlands,3,Appreciate by the ( sonic ) electronic concept...
...,...,...,...
74035,Nvidia,3,This is all based on last quarter's earnings. ...
74036,Nvidia,3,Let's see how well they handle the next one wh...
74037,Nvidia,3,Good on them. This stuff all based on earnings...
74038,Nvidia,3,9 Good idea for them. This is all based on ear...


In [15]:
df["tweet content"].duplicated().sum()

np.int64(5190)

In [16]:
df['tweet content'] =df["tweet content"].drop_duplicates()

In [17]:
df.head()

,entity,sentiment,tweet content
0,Borderlands,1,im getting on borderlands and i will murder yo...
1,Borderlands,1,I am coming to the borders and I will kill you...
2,Borderlands,1,im getting on borderlands and i will kill you ...
3,Borderlands,1,im coming on borderlands and i will murder you...
4,Borderlands,1,im getting on borderlands 2 and i will murder ...


In [18]:
df.isnull().sum()

,0
entity,0
sentiment,0
tweet content,5191


In [19]:
df=df.dropna()

In [20]:
df.isnull().sum()

,0
entity,0
sentiment,0
tweet content,0


In [21]:
def transform_text(text):
    text = nltk.word_tokenize(text.lower())

    return " ".join(
        sb.stem(word)
        for word in text
        if word.isalnum() and word not in stopwords.words('english')
    )


In [22]:
df['tweet content'] = df['tweet content'].apply(transform_text)


# RNN Model on Twitter Sentiment




In [24]:

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense , Input
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
import numpy as np


In [25]:

texts = df['tweet content'].astype(str)
labels = df['sentiment']



In [26]:
max_words = 10000
max_len = 100

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(texts)



In [28]:
sequences = tokenizer.texts_to_sequences(texts)
X = pad_sequences(sequences, maxlen=max_len)

y = to_categorical(labels)

In [29]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [32]:
model = Sequential([
    Input(shape=(max_len,)),
    Embedding(input_dim=max_words, output_dim=64 ),
    SimpleRNN(64, activation='tanh'),
    Dense(32, activation='relu'),
    Dense(y.shape[1], activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 100, 64)        │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 4)              │           132 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 650,468 (2.48 MB)

 Trainable params: 650,468 (2.48 MB)

 Non-trainable params: 0 (0.00 B)

In [34]:
history = model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2
)


Epoch 1/10
1390/1390 ━━━━━━━━━━━━━━━━━━━━ 65s 47ms/step - accuracy: 0.5947 - loss: 0.9809 - val_accuracy: 0.7145 - val_loss: 0.7488
Epoch 2/10
1390/1390 ━━━━━━━━━━━━━━━━━━━━ 56s 41ms/step - accuracy: 0.8294 - loss: 0.4739 - val_accuracy: 0.7667 - val_loss: 0.6430
Epoch 3/10
1390/1390 ━━━━━━━━━━━━━━━━━━━━ 79s 38ms/step - accuracy: 0.8897 - loss: 0.3057 - val_accuracy: 0.7746 - val_loss: 0.6799
Epoch 4/10
1390/1390 ━━━━━━━━━━━━━━━━━━━━ 55s 40ms/step - accuracy: 0.9219 - loss: 0.2179 - val_accuracy: 0.7772 - val_loss: 0.7420
Epoch 5/10
1390/1390 ━━━━━━━━━━━━━━━━━━━━ 53s 38ms/step - accuracy: 0.9399 - loss: 0.1643 - val_accuracy: 0.7677 - val_loss: 0.8160
Epoch 6/10
1390/1390 ━━━━━━━━━━━━━━━━━━━━ 56s 40ms/step - accuracy: 0.9471 - loss: 0.1431 - val_accuracy: 0.7804 - val_loss: 0.8352
Epoch 7/10
1390/1390 ━━━━━━━━━━━━━━━━━━━━ 79s 38ms/step - accuracy: 0.9438 - loss: 0.1515 - val_accuracy: 0.4821 - val_loss: 1.3915
Epoch 8/10
1390/1390 ━━━━━━━━━━━━━━━━━━━━ 84s 40ms/step - accuracy: 0.8534 -

In [36]:
loss, accuracy = model.evaluate(X_test, y_test)

print("Test Accuracy:", accuracy)


435/435 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.7865 - loss: 0.8460
Test Accuracy: 0.786531388759613


In [38]:

sample_tweet = "I really dint love this game and the graphics  are low quality"

sample_seq = tokenizer.texts_to_sequences([sample_tweet])
sample_pad = pad_sequences(sample_seq, maxlen=max_len)

prediction = model.predict(sample_pad)

predicted_class = np.argmax(prediction)

sentiment_map = {
    0: "Negative",
    1: "Positive",
    2: "Neutral",
    3: "Irrelevant"
}

print("Tweet:", sample_tweet)
print("Predicted Sentiment:", sentiment_map[predicted_class])


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step
Tweet: I really dint love this game and the graphics  are low quality
Predicted Sentiment: Irrelevant
